#### Preparation

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torchvision.models import efficientnet_v2_s, EfficientNet_V2_S_Weights
from torch.utils.data import DataLoader
from PIL import Image
import os
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from tqdm import tqdm

In [ ]:
DATA_DIR = r"C:\Users\johnp\Desktop\Dataset\Cassava_Leaf_Datasets\Classification"

WEIGHTS_DIR = r'../weights'

BATCH_SIZE = 32
FREEZE_EPOCHS = 8
FINETUNE_EPOCHS = 20

LR = 1e-4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


In [3]:
weights = EfficientNet_V2_S_Weights.DEFAULT

mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=mean,
        std=std
    )
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=mean,
        std=std
    )
])


In [4]:
train_dataset = datasets.ImageFolder(
    os.path.join(DATA_DIR, "train"),
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    os.path.join(DATA_DIR, "val"),
    transform=val_transform
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

class_names = train_dataset.classes
num_classes = len(class_names)

print("Classes:", class_names)


Classes: ['Healthy', 'Spider Mites', 'leaf blight', 'leaf spot']


In [5]:
model = efficientnet_v2_s(weights=weights)


In [6]:
for param in model.parameters():
    param.requires_grad = False


In [7]:
in_features = model.classifier[1].in_features

model.classifier[1] = nn.Linear(in_features, num_classes)

model = model.to(DEVICE)

In [8]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.classifier.parameters(),
    lr=LR
)


In [9]:
def train_one_epoch(loader, epoch, epochs):
    model.train()
    
    train_loss = 0
    train_preds = []
    train_labels = []

    for images, labels in tqdm(loader, desc=f"Training Epoch {epoch+1}/{epochs}"):
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        preds = outputs.argmax(dim=1)

        train_preds.extend(preds.cpu().numpy())
        train_labels.extend(labels.cpu().numpy())

        train_acc = accuracy_score(train_labels, train_preds)


    return train_loss / len(loader), train_acc


In [10]:
def validate(loader):
    model.eval()
    
    val_loss = 0
    val_preds = []
    val_labels = []

    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Validating"):
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()
            preds = outputs.argmax(dim=1)

            val_preds.extend(preds.cpu().numpy())
            val_labels.extend(labels.cpu().numpy())
        
        val_acc = accuracy_score(val_labels, val_preds)

        precision, recall, f1, _ = precision_recall_fscore_support(
        val_labels, val_preds, average='weighted'
        )   

    return precision, recall, f1, val_loss / len(loader), val_acc

#### Training 1 (Freeze Backbone)

In [11]:

print("\n-----------Starting Phase 1 Training-----------\n")

for epoch in range(FREEZE_EPOCHS):
    train_loss, train_acc = train_one_epoch(train_loader, epoch, FREEZE_EPOCHS)
    precision, recall, f1, val_loss, val_acc = validate(val_loader)

    print(f"\nEpoch [{epoch+1}/{FREEZE_EPOCHS}]")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
    print(f"Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}\n\n")




-----------Starting Phase 1 Training-----------



Validating: 100%|██████████| 8/8 [00:41<00:00,  5.21s/it]



Epoch [1/8]
Train Loss: 1.0509 | Train Acc: 0.6406
Val Loss: 0.8335 | Val Acc: 0.8042
Precision: 0.8059 | Recall: 0.8042 | F1: 0.8024




Validating: 100%|██████████| 8/8 [00:40<00:00,  5.04s/it]



Epoch [2/8]
Train Loss: 0.6576 | Train Acc: 0.8406
Val Loss: 0.6750 | Val Acc: 0.8167
Precision: 0.8184 | Recall: 0.8167 | F1: 0.8165




...


Epoch [6/8]
Train Loss: 0.4059 | Train Acc: 0.8833
Val Loss: 0.5183 | Val Acc: 0.8458
Precision: 0.8493 | Recall: 0.8458 | F1: 0.8457




Validating: 100%|██████████| 8/8 [00:40<00:00,  5.07s/it]



Epoch [7/8]
Train Loss: 0.3981 | Train Acc: 0.8854
Val Loss: 0.5076 | Val Acc: 0.8458
Precision: 0.8467 | Recall: 0.8458 | F1: 0.8459




Validating: 100%|██████████| 8/8 [00:39<00:00,  4.99s/it]


Epoch [8/8]
Train Loss: 0.3895 | Train Acc: 0.8781
Val Loss: 0.5106 | Val Acc: 0.8583
Precision: 0.8615 | Recall: 0.8583 | F1: 0.8581




#### Training 2 (Fine-Tuning)

In [12]:
for param in model.features[-1].parameters():
    param.requires_grad = True


In [ ]:
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-5
)


In [ ]:
best_val_acc = 0.0

print("\n-----------Starting Fine-tuning Stage-----------\n")

for epoch in range(FINETUNE_EPOCHS):
    train_loss, train_acc = train_one_epoch(train_loader, epoch, FINETUNE_EPOCHS)
    precision, recall, f1, val_loss, val_acc = validate(val_loader)

    print(f"\nEpoch [{epoch+1}/{FINETUNE_EPOCHS}]")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
    print(f"Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}\n\n")


torch.save({
    "model_state_dict": model.state_dict(),
    "class_names": class_names
}, os.path.join(WEIGHTS_DIR, "EfficientNet-v2.pth"))


-----------Starting Fine-tuning Stage-----------



Validating: 100%|██████████| 8/8 [00:37<00:00,  4.74s/it]



Epoch [1/20]
Train Loss: 0.3712 | Train Acc: 0.8802
Val Loss: 0.4989 | Val Acc: 0.8333
Precision: 0.8368 | Recall: 0.8333 | F1: 0.8323




Validating: 100%|██████████| 8/8 [00:38<00:00,  4.80s/it]



Epoch [2/20]
Train Loss: 0.3502 | Train Acc: 0.8948
Val Loss: 0.4902 | Val Acc: 0.8417
Precision: 0.8443 | Recall: 0.8417 | F1: 0.8413




...


Epoch [18/20]
Train Loss: 0.2452 | Train Acc: 0.9177
Val Loss: 0.3617 | Val Acc: 0.8667
Precision: 0.8689 | Recall: 0.8667 | F1: 0.8672




Validating: 100%|██████████| 8/8 [00:38<00:00,  4.80s/it]



Epoch [19/20]
Train Loss: 0.2497 | Train Acc: 0.9135
Val Loss: 0.3627 | Val Acc: 0.8667
Precision: 0.8688 | Recall: 0.8667 | F1: 0.8672




Validating: 100%|██████████| 8/8 [00:38<00:00,  4.83s/it]


Epoch [20/20]
Train Loss: 0.2179 | Train Acc: 0.9281
Val Loss: 0.3717 | Val Acc: 0.8750
Precision: 0.8763 | Recall: 0.8750 | F1: 0.8754




#### Prediction Sample

In [17]:
checkpoint = torch.load(r'../weights/EfficientNet-v2.pth')

model.load_state_dict(checkpoint["model_state_dict"])
class_names = checkpoint["class_names"]

model.eval()

def predict(image_path):
    image = Image.open(image_path).convert("RGB")
    image = val_transform(image).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        outputs = model(image)
        _, pred = torch.max(outputs, 1)

    return class_names[pred.item()]

# Example usage:
prediction = predict(r"C:\Users\johnp\Desktop\Dataset\Cassava_Leaf_Datasets\Classification\val\Healthy\Healthy_val_17.jpg")
prediction1 = predict(r"C:\Users\johnp\Desktop\Dataset\Cassava_Leaf_Datasets\Classification\val\Leaf Blight\leaf blight_val_17.jpg")
prediction2 = predict(r"C:\Users\johnp\Desktop\Dataset\Cassava_Leaf_Datasets\Classification\val\Leaf Spot\leaf spot_val_17.jpg")
prediction3 = predict(r"C:\Users\johnp\Desktop\Dataset\Cassava_Leaf_Datasets\Classification\val\Spider Mites\Spider Mites_val_17.jpg")
print(f"Healthy Predicted class: {prediction}")
print(f"Leaf Blight Predicted class: {prediction1}")
print(f"Leaf Spot Predicted class: {prediction2}")
print(f"Spider Mites Predicted class: {prediction3}")

Healthy Predicted class: Healthy
Leaf Blight Predicted class: leaf blight
Leaf Spot Predicted class: leaf spot
Spider Mites Predicted class: Spider Mites
